# Recommendation Engine
Fast TF-IDF based course recommender (replaces sentence-transformers which timed out).
Uses cosine similarity between skill gap text and course titles — runs in seconds locally.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

gaps_df = pd.read_csv('../data/processed/employee_skill_gaps.csv')
print(f'Loaded skill gaps: {gaps_df.shape}')
print(gaps_df.head(3))

# Course catalog with descriptive text for better TF-IDF matching
courses = [
    ('Leadership and Influence', 'leadership management influence team direction strategy'),
    ('Advanced SQL for Data Science', 'SQL database query analytics data science reporting'),
    ('Agile Project Management', 'agile project management scrum sprint planning coordination'),
    ('Machine Learning Engineering', 'machine learning engineering algorithms model deployment'),
    ('Cloud Infrastructure (AWS/Azure/GCP)', 'cloud infrastructure AWS Azure GCP DevOps servers'),
    ('Python for Data Analysis', 'python pandas data analysis programming scripting'),
    ('Strategic Communication', 'communication speaking active listening writing presentation reading comprehension'),
    ('Team Collaboration', 'teamwork collaboration conflict resolution interpersonal coordination'),
    ('Financial Literacy for Managers', 'finance budgeting accounting financial literacy P&L'),
    ('Cybersecurity Fundamentals', 'cybersecurity security network risk compliance'),
    ('Data Visualization with Tableau', 'visualization tableau charts dashboards reporting BI'),
    ('Docker and Kubernetes', 'docker kubernetes containers deployment orchestration devops'),
    ('Critical Thinking', 'critical thinking problem solving analysis reasoning judgment'),
    ('Customer Success Management', 'customer success CRM relationship management sales service'),
    ('Deep Learning with PyTorch', 'deep learning neural networks pytorch AI model training'),
    ('Business Intelligence Reporting', 'business intelligence BI reporting analytics metrics KPI')
]

course_names = [c[0] for c in courses]
course_texts = [c[1] for c in courses]

vectorizer = TfidfVectorizer()
course_matrix = vectorizer.fit_transform(course_texts)
print(f'Course catalog encoded: {course_matrix.shape}')

Loaded skill gaps: (500, 4)
   EmployeeID          JobRole  \
0           1          Auditor   
1           2  Sales Executive   
2           3         Helpdesk   

                                      current_skills  \
0  1099 ProsSoftware, Abacus Tax, Accountants Tem...   
1  Adobe Creative Cloud software, Adobe Illustrat...   
2                               General Office Tools   

                                      MissingSkills  
0  Active Listening, Reading Comprehension, Writing  
1  Active Listening, Reading Comprehension, Writing  
2                           No skill gap identified  
Course catalog encoded: (16, 91)


In [2]:
def recommend_course(skill_gap_str):
    if not skill_gap_str or str(skill_gap_str) == 'No skill gap identified':
        return 'Recommended Course: Leadership and Influence'
    query_vec = vectorizer.transform([str(skill_gap_str)])
    scores = cosine_similarity(query_vec, course_matrix)[0]
    best_idx = int(np.argmax(scores))
    return f'Recommended Course: {course_names[best_idx]}'

gaps_df['Recommendation'] = gaps_df['MissingSkills'].apply(recommend_course)
gaps_df.to_csv('../data/processed/employee_recommendations.csv', index=False)

print(f'Recommendations generated for {len(gaps_df)} employees.')
print()
print('Top recommended courses:')
print(gaps_df['Recommendation'].value_counts().head(8))
print()
print(gaps_df[['EmployeeID', 'JobRole', 'MissingSkills', 'Recommendation']].head(8).to_string(index=False))

Recommendations generated for 500 employees.

Top recommended courses:
Recommendation
Recommended Course: Strategic Communication     420
Recommended Course: Leadership and Influence     80
Name: count, dtype: int64

 EmployeeID         JobRole                                    MissingSkills                               Recommendation
          1         Auditor Active Listening, Reading Comprehension, Writing  Recommended Course: Strategic Communication
          2 Sales Executive Active Listening, Reading Comprehension, Writing  Recommended Course: Strategic Communication
          3        Helpdesk                          No skill gap identified Recommended Course: Leadership and Influence
          4    HR Executive Active Listening, Reading Comprehension, Writing  Recommended Course: Strategic Communication
          5 Account Manager Active Listening, Reading Comprehension, Writing  Recommended Course: Strategic Communication
          6        Helpdesk                        